In [6]:
import sys
import os
from pathlib import Path

project_root = Path("/content/drive/MyDrive/CCTV_VLM_Project")

if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

os.chdir(project_root)

In [7]:
import time
import cv2
import numpy as np
import pandas as pd
import supervision as sv
from pathlib import Path
from trackers import ByteTrackTracker

In [8]:
from src.config import(
    MODELS_DIR, VIDEOS_DIR, OUTPUTS_DIR,
    CLASS_MAPPING, YOLO_CONF_THRESH, NMS_IOU_THRESH,
    DINO_PROMPT, DINO_BOX_THRESH, DINO_TEXT_THRESH
)

from src.yolo_detector import YoloDetector
from src.dino_detector import DinoDetector
from src.fusion import BoxFusion

yolo = YoloDetector(str(MODELS_DIR / "yolo11s_warehouse_best.pt") , conf = YOLO_CONF_THRESH)
dino = DinoDetector(box_thresh = DINO_BOX_THRESH , text_thresh = DINO_TEXT_THRESH)
fusion = BoxFusion(iou_threshold=NMS_IOU_THRESH)

Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings


/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


preprocessor_config.json:   0%|          | 0.00/457 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.64k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.24k [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

... (2 outputs left out)

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  689MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/978 [00:00<?, ?it/s]

In [9]:
video_files = [str(VIDEOS_DIR/ f"warehouse{i}.mp4") for i in [1 , 2,  3, 4]]
all_tracking_records = []

box_annotator = sv.BoxAnnotator(thickness = 2)
label_annotator = sv.LabelAnnotator(text_scale = 0.5 , text_thickness =1)
trace_annotator = sv.TraceAnnotator(thickness = 2 , trace_length= 40 , position=sv.Position.BOTTOM_CENTER)

print("🚀 Starting Batch Run Across All 4 Videos with Tuned ByteTrackTracker...")

for v_path in video_files:
    video_path_obj = Path(v_path)
    if not video_path_obj.exists():
        print(f"⚠️ Video not found: {v_path}")
        continue

    video_name = video_path_obj.stem
    video_info = sv.VideoInfo.from_video_path(v_path)
    fps = int(round(video_info.fps)) or 25
    W, H = int(video_info.width), int(video_info.height)
    out_video_path = str(OUTPUTS_DIR / f"{video_name}_tracked_verified.mp4")

    tracker = ByteTrackTracker(
        track_activation_threshold = 0.30,
        lost_track_buffer = 50,
        frame_rate = fps,
        minimum_iou_threshold = 0.5,
        minimum_consecutive_frames = 5,
        high_conf_det_threshold = 0.3
    )

    print(f"\n🎬 Processing {video_name} ({W}x{H} @ {fps} fps)...")
    start_time = time.time()
    video_records_count = 0

    with sv.VideoSink(target_path = out_video_path , video_info = video_info , codec = "mp4v") as sink:

        for frame_idx, frame in enumerate(sv.get_video_frames_generator(source_path=v_path)):
            timestamp = round(frame_idx / fps, 3)

            # 1. YOLO + Live DINO
            yolo_det = yolo.detect(frame)
            dino_det = dino.detect(frame, prompt=DINO_PROMPT)

            # 2. Fuse outputs
            fused_det = fusion.merge_and_suppress(yolo_det, dino_det)

            annotated = frame.copy()

            if len(fused_det) == 0:
                sink.write_frame(annotated)
                continue

            # 3. Update Tracker
            tracked_det = tracker.update(detections=fused_det)

            # 4. Associate Tracked IDs back to Fused Detections (Preserves true class_id)
            tracked_by_index = {}
            used_indices = set()

            if len(tracked_det) > 0 and tracked_det.tracker_id is not None:
                # Find matching fused index for each tracked box
                for box, track_id in zip(tracked_det.xyxy, tracked_det.tracker_id):
                    # Coordinate difference check with IoU fallback
                    diffs = np.max(np.abs(fused_det.xyxy - box), axis=1)
                    available = np.array([i not in used_indices for i in range(len(fused_det))])
                    candidates = np.where((diffs <= 1.0) & available)[0]

                    if len(candidates) > 0:
                        match_idx = int(candidates[0])
                        tracked_by_index[match_idx] = int(track_id)
                        used_indices.add(match_idx)

            # 5. Extract and Log Each Detection
            valid_tracked_boxes = []
            valid_tracked_ids = []
            valid_class_ids = []
            valid_labels = []

            for det_idx, (box, confidence, class_id) in enumerate(
                zip(fused_det.xyxy, fused_det.confidence, fused_det.class_id)
            ):
                track_id = tracked_by_index.get(det_idx, -1)
                cid = int(class_id)
                cname = CLASS_MAPPING.get(cid, "unknown")
                conf = float(confidence)

                x1, y1, x2, y2 = box
                cx = (x1 + x2) / 2.0
                cy = (y1 + y2) / 2.0
                feet_x = cx
                feet_y = y2  # Floor contact point

                all_tracking_records.append({
                    "video": video_name,
                    "frame_idx": frame_idx,
                    "timestamp_sec": timestamp,
                    "track_id": track_id,
                    "class_id": cid,
                    "class_name": cname,
                    "confidence": round(conf, 3),
                    "x1": round(float(x1), 1),
                    "y1": round(float(y1), 1),
                    "x2": round(float(x2), 1),
                    "y2": round(float(y2), 1),
                    "w": round(float(x2 - x1), 1),
                    "h": round(float(y2 - y1), 1),
                    "aspect_ratio": round(float((x2 - x1) / max(1.0, (y2 - y1))), 3),
                    "cx": round(float(cx), 1),
                    "cy": round(float(cy), 1),
                    "feet_x": round(float(feet_x), 1),
                    "feet_y": round(float(feet_y), 1),
                    "frame_w": W,
                    "frame_h": H,
                    "norm_cx": round(float(cx / W), 4),
                    "norm_cy": round(float(cy / H), 4),
                    "norm_feet_x": round(float(feet_x / W), 4),
                    "norm_feet_y": round(float(feet_y / H), 4),
                    "norm_w": round(float((x2 - x1) / W), 4),
                    "norm_h": round(float((y2 - y1) / H), 4)
                })
                video_records_count += 1

                # Prep annotation arrays for confirmed tracks
                if track_id >= 0:
                    valid_tracked_boxes.append(box)
                    valid_tracked_ids.append(track_id)
                    valid_class_ids.append(cid)
                    valid_labels.append(f"#{track_id} {cname} {conf:.2f}")

            # 6. Render Traces & Boxes on Frame
            if len(valid_tracked_boxes) > 0:
                draw_det = sv.Detections(
                    xyxy=np.array(valid_tracked_boxes, dtype=np.float32),
                    confidence=np.ones(len(valid_tracked_boxes), dtype=np.float32),
                    class_id=np.array(valid_class_ids, dtype=int),
                    tracker_id=np.array(valid_tracked_ids, dtype=int)
                )
                annotated = trace_annotator.annotate(scene=annotated, detections=draw_det)
                annotated = box_annotator.annotate(scene=annotated, detections=draw_det)
                annotated = label_annotator.annotate(scene=annotated, detections=draw_det, labels=valid_labels)

            sink.write_frame(annotated)

    elapsed = round(time.time() - start_time, 1)
    print(f"   Done {video_name} ({video_records_count} detections logged in {elapsed}s) -> Saved: {out_video_path}")

# 7. Compile Final CSV
df_master = pd.DataFrame(all_tracking_records)
csv_save_path = OUTPUTS_DIR / "fused_bytetrack_detections.csv"
df_master.to_csv(csv_save_path, index=False)

print("\n" + "="*60)
print(f"✅ All 4 Videos Processed!")
print(f"📁 Master CSV saved to: {csv_save_path} ({len(df_master)} total records)")
print("="*60)

    

🚀 Starting Batch Run Across All 4 Videos with Tuned ByteTrackTracker...

🎬 Processing warehouse1 (1920x1080 @ 25 fps)...
   Done warehouse1 (1073 detections logged in 130.8s) -> Saved: /content/drive/MyDrive/CCTV_VLM_Project/outputs/warehouse1_tracked_verified.mp4

🎬 Processing warehouse2 (1920x1080 @ 25 fps)...
   Done warehouse2 (953 detections logged in 113.6s) -> Saved: /content/drive/MyDrive/CCTV_VLM_Project/outputs/warehouse2_tracked_verified.mp4

🎬 Processing warehouse3 (1920x1080 @ 25 fps)...
   Done warehouse3 (901 detections logged in 129.7s) -> Saved: /content/drive/MyDrive/CCTV_VLM_Project/outputs/warehouse3_tracked_verified.mp4

🎬 Processing warehouse4 (1920x1080 @ 25 fps)...
   Done warehouse4 (1037 detections logged in 113.7s) -> Saved: /content/drive/MyDrive/CCTV_VLM_Project/outputs/warehouse4_tracked_verified.mp4

✅ All 4 Videos Processed!
📁 Master CSV saved to: /content/drive/MyDrive/CCTV_VLM_Project/outputs/fused_bytetrack_detections.csv (3964 total records)


In [10]:
# 1. Total detections per class
print("📊 Total Detections by Class Across All Videos:")
display(df_master["class_name"].value_counts())

# 2. Confirmed tracks (track_id >= 0) per video and class
df_confirmed = df_master[df_master["track_id"] >= 0]
print("\n🎯 Unique Confirmed Tracks per Video and Class:")
summary = df_confirmed.groupby(["video", "class_name"])["track_id"].nunique().reset_index(name="unique_tracks")
display(summary)

# 3. Quick check on track lifetimes
print("\n⏱️ Top 10 Longest-Lived Tracks:")
track_lifetimes = df_confirmed.groupby(["video", "track_id", "class_name"]).agg(
    frames_tracked=("frame_idx", "count"),
    start_sec=("timestamp_sec", "min"),
    end_sec=("timestamp_sec", "max")
).reset_index().sort_values(by="frames_tracked", ascending=False)
display(track_lifetimes.head(10))


📊 Total Detections by Class Across All Videos:


,count
class_name,
person,2149
pallet_truck,1532
forklift,283



🎯 Unique Confirmed Tracks per Video and Class:


,video,class_name,unique_tracks
0,warehouse1,forklift,2
1,warehouse1,pallet_truck,3
2,warehouse1,person,8
3,warehouse2,forklift,1
4,warehouse2,pallet_truck,2
5,warehouse2,person,13
6,warehouse3,forklift,3
7,warehouse3,pallet_truck,3
8,warehouse3,person,6
9,warehouse4,pallet_truck,2



⏱️ Top 10 Longest-Lived Tracks:


,video,track_id,class_name,frames_tracked,start_sec,end_sec
0,warehouse1,0,pallet_truck,194,0.16,7.88
31,warehouse3,2,pallet_truck,194,0.16,7.88
29,warehouse3,0,pallet_truck,194,0.16,7.88
1,warehouse1,1,pallet_truck,184,0.16,7.88
43,warehouse4,2,person,169,0.16,6.88
15,warehouse2,2,person,169,0.16,6.88
18,warehouse2,4,pallet_truck,169,0.16,6.88
41,warehouse4,0,person,169,0.16,6.88
44,warehouse4,3,pallet_truck,167,0.16,6.88
17,warehouse2,3,pallet_truck,155,0.16,6.88


In [22]:
from pathlib import Path

PROJECT_ROOT = Path("/content/drive/MyDrive/CCTV_VLM_Project")

print("VISION SENTINEL PROJECT")
print("=" * 55)

for path in sorted(PROJECT_ROOT.iterdir()):
    if path.is_dir():
        print(f"\n📁 {path.name}/")
        for child in sorted(path.iterdir())[:20]:
            prefix = "📁" if child.is_dir() else "📄"
            print(f"   {prefix} {child.name}")
    else:
        print(f"📄 {path.name}")

VISION SENTINEL PROJECT

📁 .ipynb_checkpoints/

📁 models/
   📄 yolo11n_E3_PPE_best.pt
   📄 yolo11s_warehouse_best.pt
   📄 yolo26s_warehouse_best.pt

📁 notebooks/
   📁 .ipynb_checkpoints
   📄 check_yolo_model.ipynb
   📄 fused_model.ipynb
   📄 run_production.ipynb
   📄 scene_zone_experiment.ipynb
   📄 tracker_experimenet_1.ipynb
   📄 tracker_new_exp.ipynb

📁 outputs/
   📁 .ipynb_checkpoints
   📄 fused_bytetrack_detections.csv
   📄 warehouse1_tracked_verified.mp4
   📄 warehouse2_tracked_verified.mp4
   📄 warehouse3_tracked_verified.mp4
   📄 warehouse4_tracked_verified.mp4
📄 requirements.txt

📁 src/
   📄 __init__.py
   📁 __pycache__
   📄 config.py
   📄 dino_detector.py
   📄 fusion.py
   📄 main.py
   📄 yolo_detector.py

📁 videos/
   📁 .ipynb_checkpoints
   📁 camera001

📁 warehouse_run_artifacts/
   📄 BoxF1_curve.png
   📄 BoxPR_curve.png
   📄 BoxP_curve.png
   📄 BoxR_curve.png
   📄 args.yaml
   📄 confusion_matrix.png
   📄 confusion_matrix_normalized.png
   📄 labels.jpg
   📄 results.csv
   📄 